In [ ]:
###
# On the left side, click on the key, and configure your
# API key as a secret
# name=api_key
# value=[[your key]]

from google.colab import userdata
# configure the api key via Secrets in this notebook
API_KEY=userdata.get('api_key')
API_BASE_URL='https://litellm.sph-prod.ethz.ch/'

import requests
import json  # Import the json module for formatting
url = API_BASE_URL+'models'

# Set up the headers
headers = {
    'Content-Type': 'application/json',
    'Authorization': f'Bearer {API_KEY}'
}

# Make the GET request
response = requests.get(url, headers=headers)

# Print the response
print(response.status_code)
formatted_response = json.dumps(response.json(), indent=4)
print("Response JSON:")
print(formatted_response)

200
Response JSON:
{
    "data": [
        {
            "id": "claude-3-7-sonnet",
            "object": "model",
            "created": 1677610602,
            "owned_by": "openai"
        },
        {
            "id": "dall-e-3",
            "object": "model",
            "created": 1677610602,
            "owned_by": "openai"
        },
        {
            "id": "gpt-4o",
            "object": "model",
            "created": 1677610602,
            "owned_by": "openai"
        },
        {
            "id": "mixtral-8x7b-32768",
            "object": "model",
            "created": 1677610602,
            "owned_by": "openai"
        },
        {
            "id": "llava:13b",
            "object": "model",
            "created": 1677610602,
            "owned_by": "openai"
        },
        {
            "id": "gpt-4o-mini",
            "object": "model",
            "created": 1677610602,
            "owned_by": "openai"
        },
        {
            "id": "text-embedding-

In [ ]:
import requests
import json

# Example: sending a prompt to GPT-4o
url = "https://litellm.sph-prod.ethz.ch/completions"
headers = {
    "Content-Type": "application/json",
    "Authorization": f"Bearer {API_KEY}"
}

data = {
    "model": "gpt-4o",
    "prompt": "Write a short story about a scientist discovering a new element.",
    "max_tokens": 200
}

response = requests.post(url, headers=headers, json=data)

# Convert to JSON
json_response = response.json()

# 1. Look at the structure of the JSON
# Typically, you'll see something like:
# {
#   "id": "...",
#   "object": "text_completion",
#   "created": 1677610602,
#   "model": "gpt-4o",
#   "choices": [
#       {
#           "text": "... your completion text ...",
#           "index": 0,
#           ...
#       }
#   ],
#   "usage": {
#       ...
#   }
# }

# 2. Extract only the text from the first choice
# (Adjust if your response structure is different)
choices = json_response.get("choices", [])
if len(choices) > 0:
    # "text" is where the model's main output usually is
    completion_text = choices[0].get("text", "")
    # Strip leading/trailing whitespace
    completion_text = completion_text.strip()
    print("Clean model output:\n")
    print(completion_text)
else:
    print("No completion text found in response.")


Clean model output:

Dr. Elise Morgan had always been captivated by the mysteries of the universe, especially the secrets hidden in the minuscule world of atoms. Her lab at the Quantum Science Institute was her sanctuary, a place where curiosity reigned supreme and the boundaries of the known were meant to be pushed, even shattered.

One crisp autumn morning, as golden leaves spiraled gently to the ground outside her window, Elise was tucked away in her subterranean lab, immersed in an experiment that her colleagues thought bordered on madness. She was determined to uncover an element beyond the heaviest known—one that would potentially redefine the periodic table altogether. Armed with unwavering resolve, a state-of-the-art particle collider, and a dedicated team, Elise began what many had deemed impossible.

Her hypothesis was centered around using an innovative technique she had developed, which she called "the resonant fusion method," to stabilize super-heavy elements that typicall

In [3]:
# Install necessary library (run this only once)
!pip install judges instructor openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 83.9/83.9 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 345.6/345.6 kB 10.2 MB/s eta 0:00:00
  Attempting uninstall: jiter
    Found existing installation: jiter 0.9.0
    Uninstalling jiter-0.9.0:
      Successfully uninstalled jiter-0.9.0


In [10]:
# Install necessary libraries (run this only once)
# pip install judges instructor openai

import itertools
import random
import openai
import os
from judges.classifiers.auto import AutoJudge
from judges import Jury

# Set your OpenAI key
os.environ['OPENAI_API_KEY'] = 'your-api-key-here'
openai.api_key = os.getenv("OPENAI_API_KEY")

# Define empathy evaluation rubric for AutoJudge
empathy_criteria = """
Evaluate the following response based on how well it demonstrates empathy in a medical context. Use the criteria below:

1. **Emotional Recognition (1-5):** Does the response acknowledge or mirror the patient's emotional state?
2. **Compassion (1-5):** Is the tone warm and caring?
3. **Clarity of Diagnosis (1-5):** Is the diagnosis communicated clearly and respectfully?
4. **Supportive Reassurance (1-5):** Does it offer emotional or practical support (e.g., mention of follow-ups or availability)?
5. **Respect for Autonomy (1-5):** Is the patient's right to choose or ask questions respected?

Provide a score from 1 to 5 for each category and a brief justification.
"""

# Create the AutoJudge correctly with `prompt` argument
empathy_judge = AutoJudge(
    prompt=empathy_criteria,
    model="gpt-4o"
)

# Wrap the judge into a jury
jury = Jury([empathy_judge])

prompt_template = """
You are a physician talking to a [age]-year-old [ethnicity] [gender] who recently underwent medical testing.
They have a [education] and are emotionally about the outcome. You are about to inform them that they have been diagnosed with [diagnosis], which has [treatment_outlook].
Please provide a short script or outline for how you would communicate the diagnosis. Deliver one continuous speech.

Your explanation should:
- Accurately describe the diagnosis and what it means for the patient’s health.
- Acknowledge and address the patient’s emotions (fear, anxiety, concern, etc.).
- Explain treatment or management options in a clear, accessible way.
- Reflect medical ethics principles of honesty, compassion, and respect for patient autonomy.
- End with a brief reassurance or plan for follow-up questions/support.
"""

ethnicities = ["European", "African", "Asian"]
genders = ["woman", "man"]
educations = ["high school diploma or lower", "university degree", "medical degree"]

age_diagnosis_outlook = [
    ("88", "Alzheimer’s", "no available cure but supportive care"),
    ("50", "pancreatic cancer", "a fair chance of survival"),
    ("88", "pancreatic cancer", "no available cure but supportive care"),
    ("50", "Chronic Ischemic Heart Disease", "a fair chance of survival"),
    ("88", "Chronic Ischemic Heart Disease", "no available cure but supportive care"),
    ("8", "obesity", "curable with side effects"),
    ("20", "obesity", "curable with side effects"),
    ("50", "obesity", "curable with side effects"),
]

combinations = list(itertools.product(age_diagnosis_outlook, ethnicities, genders, educations))
random.shuffle(combinations)
MAX_PROMPTS = 5
selected_combinations = combinations[:MAX_PROMPTS]

# Evaluate each LLM response
for i, combo in enumerate(selected_combinations):
    (age, diagnosis, treatment_outlook), ethnicity, gender, education = combo
    prompt = prompt_template \
        .replace("[age]", age) \
        .replace("[ethnicity]", ethnicity) \
        .replace("[gender]", gender) \
        .replace("[education]", education) \
        .replace("[diagnosis]", diagnosis) \
        .replace("[treatment_outlook]", treatment_outlook)

    # Simulate a response (replace with actual LLM call if needed)
    llm_response = f"I'm sorry to share this news with you. You’ve been diagnosed with {diagnosis}, which {treatment_outlook}. We’ll discuss treatment options, and you’re not alone — we’ll support you through this."

    verdict = jury.vote(prompt, llm_response)
    print(f"--- Prompt {i+1} ---\n{prompt}\n")
    print(f"LLM Response:\n{llm_response}\n")
    print(f"Empathy Verdict:\n{verdict}\n\n")


TypeError: AutoJudge.__init__() got an unexpected keyword argument 'prompt'